> **Provenance only — not runnable from the public repository; outputs intentionally cleared.**
>
> This robustness notebook compares the **raw** and **technically cleaned** cohort
> files (`data/BGA_merged_all_20260208.csv` and `…_cleaned.csv`), both of which are
> **non-public** and **not distributed** with this repository.
>
> Its cell outputs are **deliberately left empty** to avoid displaying row-level,
> non-blinded cohort data. Do **not** commit executed outputs of this notebook.
>
> The public, runnable analysis path is **`03_table_1_generation_blinded.ipynb`**,
> which uses the released blinded dataset.

# JINS analysis-ready reproducibility notebook

This notebook reproduces the manuscript-ready numerics currently reported in `manuscript/jins_main.tex` from the analysis-ready cohort file `data/BGA_merged_all_20260208_cleaned_for_analysis.csv` used by `scripts/neuropsych_pipeline.py`.

It is designed to:

- load the analysis-ready cohort used by the standalone March 16 pipeline
- regenerate the Table 1 summary statistics and inferential tests
- regenerate the pipeline summary counts reported in the Results section under the active flag rules
- regenerate the key patient-level values for the two illustrative cases
- check consistency between the computed outputs and the current `manuscript/jins_main.tex` manuscript

The notebook is intentionally explicit so that the revised numerics are documented in a durable and inspectable analysis artifact.

In [ ]:
from pathlib import Path
import importlib.util
import math
import re

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import Markdown, display


def detect_repo_root(start=None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "manuscript").exists() and (candidate / "scripts").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from current working directory.")


ROOT = detect_repo_root()
MANUSCRIPT_PATH = ROOT / "manuscript" / "jins_main.tex"
PIPELINE_PATH = ROOT / "scripts" / "neuropsych_pipeline.py"

spec = importlib.util.spec_from_file_location("pipeline", PIPELINE_PATH)
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)

DATA_PATH = Path(pipeline.DATA_PATH)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print(f"Repository root: {ROOT}")
print(f"Analysis-ready data file: {DATA_PATH}")
print(f"Manuscript file: {MANUSCRIPT_PATH}")

In [ ]:
df = pipeline.load_cohort(str(DATA_PATH))
ibs = df[df[pipeline.COL_GROUP] == "IBS"].copy()
hc = df[df[pipeline.COL_GROUP] == "HC"].copy()


def female_mask(series: pd.Series) -> pd.Series:
    return series.astype(str).str.upper().str.startswith("F")


def cohens_d_pooled(x: pd.Series, y: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce").dropna()
    y = pd.to_numeric(y, errors="coerce").dropna()
    nx, ny = len(x), len(y)
    sx, sy = x.std(ddof=1), y.std(ddof=1)
    sp = np.sqrt(((nx - 1) * sx**2 + (ny - 1) * sy**2) / (nx + ny - 2))
    return float((x.mean() - y.mean()) / sp)


def summarize_measure(label: str, col: str) -> dict:
    ibs_vals = pd.to_numeric(ibs[col], errors="coerce").dropna()
    hc_vals = pd.to_numeric(hc[col], errors="coerce").dropna()
    test = stats.ttest_ind(ibs_vals, hc_vals, equal_var=False, nan_policy="omit")
    return {
        "Measure": label,
        "IBS_n": len(ibs_vals),
        "HC_n": len(hc_vals),
        "IBS_mean": ibs_vals.mean(),
        "IBS_sd": ibs_vals.std(ddof=1),
        "HC_mean": hc_vals.mean(),
        "HC_sd": hc_vals.std(ddof=1),
        "t": float(test.statistic),
        "p": float(test.pvalue),
        "d": cohens_d_pooled(ibs_vals, hc_vals),
    }


measure_specs = [
    ("Age (years)", pipeline.COL_AGE),
    ("Education (years)", pipeline.COL_EDUCATION),
    ("BIS total", "BIS_total"),
    ("Detectability", "CPT_Detectability"),
    ("Omissions", "CPT_Omissions"),
    ("Commissions", "CPT_Commissions"),
    ("HRT", "CPT_HRT"),
    ("Chalder total", "Chalder_total"),
    ("Anxiety", "HADS_Anxiety"),
    ("Depression", "HADS_Depression"),
    ("Immediate Memory summary", pipeline.RBANS_SUMMARY_COLS["Immediate Memory"]),
    ("Visuospatial summary", pipeline.RBANS_SUMMARY_COLS["Visuospatial"]),
    ("Language summary", pipeline.RBANS_SUMMARY_COLS["Language"]),
    ("Attention summary", pipeline.RBANS_SUMMARY_COLS["Attention"]),
    ("Delayed Memory summary", pipeline.RBANS_SUMMARY_COLS["Delayed Memory"]),
    ("Total Scale (Sum Raw)", pipeline.RBANS_SUMMARY_COLS["Total Scale"]),
]

table1_stats = pd.DataFrame([summarize_measure(label, col) for label, col in measure_specs])

female_ibs = int(female_mask(ibs[pipeline.COL_GENDER]).sum())
female_hc = int(female_mask(hc[pipeline.COL_GENDER]).sum())
chi2, chi2_p, _, _ = stats.chi2_contingency([
    [female_ibs, len(ibs) - female_ibs],
    [female_hc, len(hc) - female_hc],
])

cohort_summary = {
    "shape": df.shape,
    "group_counts": df[pipeline.COL_GROUP].value_counts(dropna=False).to_dict(),
    "flag_columns": [c for c in df.columns if "flag" in c.lower() or "issue" in c.lower()],
    "female_pct_ibs": round(female_ibs / len(ibs) * 100, 1),
    "female_pct_hc": round(female_hc / len(hc) * 100, 1),
    "female_pct_total": round(female_mask(df[pipeline.COL_GENDER]).mean() * 100, 1),
    "ibs_subtypes": ibs[pipeline.COL_IBS_TYPE].value_counts(dropna=False).to_dict(),
    "ibs_sss_n": int(pd.to_numeric(ibs[pipeline.COL_IBS_SSS], errors="coerce").notna().sum()),
    "ibs_sss_mean": round(float(pd.to_numeric(ibs[pipeline.COL_IBS_SSS], errors="coerce").mean()), 1),
    "ibs_sss_sd": round(float(pd.to_numeric(ibs[pipeline.COL_IBS_SSS], errors="coerce").std(ddof=1)), 1),
    "gender_chi2": float(chi2),
    "gender_p": float(chi2_p),
}

print(cohort_summary)
display(table1_stats.round(4))

In [ ]:
all_results = {
    pid: pipeline.analyze_patient(df, pid)
    for pid in df[pipeline.COL_SUBJECT].tolist()
}

case_ids = ["subj_001", "subj_048"]
case_results = {pid: all_results[pid] for pid in case_ids}

pipeline_counts = {
    "sleep_total": 0,
    "sleep_ibs": 0,
    "fatigue_total": 0,
    "fatigue_ibs": 0,
    "hads_anxiety_total": 0,
    "hads_anxiety_ibs": 0,
    "hads_depression_total": 0,
    "hads_depression_ibs": 0,
    "attention_total": 0,
    "attention_ibs": 0,
    "cognition_total": 0,
    "cognition_ibs": 0,
    "multi_domain_total": 0,
    "multi_domain_ibs": 0,
}

for pid, result in all_results.items():
    domains = result["domain_findings"]
    group = result["demographics"]["ibs_status"]

    sleep_flag = bool(domains["sleep_BIS"]["flags"])
    fatigue_flag = bool(domains["fatigue_Chalder"]["flags"])
    mood_flag = bool(domains["emotional_distress_HADS"]["flags"])
    attention_flag = bool(domains["attention_CPT"]["flags"])
    cognition_flag = bool(domains["neurocognition_RBANS"]["flags"])
    anxiety_flag = any("HADS-A" in flag for flag in domains["emotional_distress_HADS"]["flags"])
    depression_flag = any("HADS-D" in flag for flag in domains["emotional_distress_HADS"]["flags"])
    flagged_domain_count = sum([sleep_flag, fatigue_flag, mood_flag, attention_flag, cognition_flag])

    if sleep_flag:
        pipeline_counts["sleep_total"] += 1
        if group == "IBS":
            pipeline_counts["sleep_ibs"] += 1
    if fatigue_flag:
        pipeline_counts["fatigue_total"] += 1
        if group == "IBS":
            pipeline_counts["fatigue_ibs"] += 1
    if anxiety_flag:
        pipeline_counts["hads_anxiety_total"] += 1
        if group == "IBS":
            pipeline_counts["hads_anxiety_ibs"] += 1
    if depression_flag:
        pipeline_counts["hads_depression_total"] += 1
        if group == "IBS":
            pipeline_counts["hads_depression_ibs"] += 1
    if attention_flag:
        pipeline_counts["attention_total"] += 1
        if group == "IBS":
            pipeline_counts["attention_ibs"] += 1
    if cognition_flag:
        pipeline_counts["cognition_total"] += 1
        if group == "IBS":
            pipeline_counts["cognition_ibs"] += 1
    if flagged_domain_count >= 2:
        pipeline_counts["multi_domain_total"] += 1
        if group == "IBS":
            pipeline_counts["multi_domain_ibs"] += 1

print(pipeline_counts)
for pid, result in case_results.items():
    print("\n", pid)
    print(result["demographics"])
    for key in ["sleep_BIS", "fatigue_Chalder", "emotional_distress_HADS", "attention_CPT", "neurocognition_RBANS"]:
        print(key, result["domain_findings"][key])

In [ ]:
def p_stars(p: float) -> str:
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def fmt_signed(value: float, digits: int = 2) -> str:
    rounded = round(float(value), digits)
    if rounded == 0:
        rounded = 0.0
    if rounded < 0:
        return f"$-{abs(rounded):.{digits}f}$"
    return f"{rounded:.{digits}f}"


def fmt_mean(value: float, digits: int = 1) -> str:
    rounded = round(float(value), digits)
    if rounded == 0:
        rounded = 0.0
    return f"{rounded:.{digits}f}"


def fmt_mean_sd(mean: float, sd: float) -> str:
    return f"{fmt_mean(mean)} ({fmt_mean(sd)})"


def row_lookup(label: str) -> pd.Series:
    row = table1_stats.loc[table1_stats["Measure"] == label]
    if row.empty:
        raise KeyError(label)
    return row.iloc[0]


def make_row(label: str, measure_label: str, left_suffix: str = "") -> str:
    row = row_lookup(measure_label)
    return (
        f"\\quad {label}{left_suffix} & "
        f"{fmt_mean_sd(row['IBS_mean'], row['IBS_sd'])} & "
        f"{fmt_mean_sd(row['HC_mean'], row['HC_sd'])} & "
        f"{fmt_signed(row['t'])}{p_stars(row['p'])} & {fmt_signed(row['d'])} \\\\"
    )


manuscript_rows = {
    "Age (years)": make_row("Age (years)", "Age (years)"),
    "Education (years)": make_row("Education (years)", "Education (years)", left_suffix="$^{a}$"),
    "BIS total": make_row("BIS total (0--42)", "BIS total"),
    "Detectability": make_row("Detectability (d')", "Detectability"),
    "Chalder total": make_row("Chalder total (0--11)", "Chalder total"),
    "Anxiety": make_row("Anxiety (0--21)", "Anxiety"),
    "Depression": make_row("Depression (0--21)", "Depression"),
    "Immediate Memory summary": make_row("Immediate Memory summary", "Immediate Memory summary"),
    "Attention summary": make_row("Attention summary", "Attention summary"),
    "Delayed Memory summary": make_row("Delayed Memory summary", "Delayed Memory summary"),
    "Total Scale (Sum Raw)": make_row("Total Scale (Sum Raw)", "Total Scale (Sum Raw)"),
}

abstract_summary = {
    "sleep_pct": round(pipeline_counts['sleep_total'] / len(df) * 100),
    "fatigue_pct": round(pipeline_counts['fatigue_total'] / len(df) * 100),
    "anxiety_pct": round(pipeline_counts['hads_anxiety_total'] / len(df) * 100),
    "attention_pct": round(pipeline_counts['attention_total'] / len(df) * 100),
    "multi_pct": round(pipeline_counts['multi_domain_total'] / len(df) * 100),
}

results_summary_md = f"""
## Manuscript-ready March 16 numerics

- Cohort: `N = {df.shape[0]}` participants (`{cohort_summary['group_counts']['IBS']}` IBS, `{cohort_summary['group_counts']['HC']}` HC)
- Female proportion: `{cohort_summary['female_pct_total']}%`
- IBS subtypes: `D={cohort_summary['ibs_subtypes']['D']}`, `C={cohort_summary['ibs_subtypes']['C']}`, `M={cohort_summary['ibs_subtypes']['M']}`
- IBS-SSS available for `n = {cohort_summary['ibs_sss_n']}` IBS patients, `M = {cohort_summary['ibs_sss_mean']}`, `SD = {cohort_summary['ibs_sss_sd']}`

### Active pipeline summary counts

- Sleep-domain flags: `{pipeline_counts['sleep_total']}` total, `{pipeline_counts['sleep_ibs']}` IBS
- Fatigue caseness: `{pipeline_counts['fatigue_total']}` total, `{pipeline_counts['fatigue_ibs']}` IBS
- HADS-A borderline/clinical: `{pipeline_counts['hads_anxiety_total']}` total, `{pipeline_counts['hads_anxiety_ibs']}` IBS
- HADS-D borderline/clinical: `{pipeline_counts['hads_depression_total']}` total, `{pipeline_counts['hads_depression_ibs']}` IBS
- Flagged CPT profiles: `{pipeline_counts['attention_total']}` total
- Flagged RBANS profiles: `{pipeline_counts['cognition_total']}` total, `{pipeline_counts['cognition_ibs']}` IBS
- Multi-domain elevations: `{pipeline_counts['multi_domain_total']}` total, `{pipeline_counts['multi_domain_ibs']}` IBS

### Abstract-level percentages

- Sleep: `{abstract_summary['sleep_pct']}%`
- Fatigue: `{abstract_summary['fatigue_pct']}%`
- Anxiety: `{abstract_summary['anxiety_pct']}%`
- Attention: `{abstract_summary['attention_pct']}%`
- Multi-domain elevations: `{abstract_summary['multi_pct']}%`
"""

display(Markdown(results_summary_md))

In [ ]:
tex = MANUSCRIPT_PATH.read_text()

checks = {
    "analysis-ready file named in Methods": (
        "BGA\\_merged\\_all\\_20260208\\_cleaned\\_for\\_analysis.csv" in tex
        or "\\path{BGA_merged_all_20260208_cleaned_for_analysis.csv}" in tex
    ),
    "Table 1 Age row": "\\quad Age (years) & 37.8 (11.4) & 35.6 (12.5) & 0.90 & 0.19 \\\\" in tex,
    "Table 1 Education row": "\\quad Education (years)$^{a}$ & 15.8 (1.9) & 16.4 (1.4) & $-$1.54 & $-$0.31 \\\\" in tex,
    "Table 1 BIS row": "\\quad BIS total (0--42) & 17.6 (7.6) & 10.3 (6.9) & 4.89*** & 0.99 \\\\" in tex,
    "Table 1 Detectability row": "\\quad Detectability (d') & 48.9 (7.8) & 44.3 (7.1) & 3.00** & 0.60 \\\\" in tex,
    "Table 1 Chalder row": "\\quad Chalder total (0--11) & 6.4 (3.4) & 1.6 (2.5) & 7.37*** & 1.55 \\\\" in tex,
    "Table 1 Anxiety row": "\\quad Anxiety (0--21) & 8.1 (4.2) & 4.2 (3.3) & 4.97*** & 1.00 \\\\" in tex,
    "Table 1 Depression row": "\\quad Depression (0--21) & 4.7 (3.1) & 2.1 (2.3) & 4.52*** & 0.90 \\\\" in tex,
    "Table 1 Immediate Memory summary row": "\\quad Immediate Memory summary & $-$0.2 (0.8) & 0.3 (0.7) & $-$2.94** & $-$0.59 \\\\" in tex,
    "Table 1 Attention summary row": "\\quad Attention summary & $-$0.2 (0.7) & 0.3 (0.9) & $-$2.67** & $-$0.60 \\\\" in tex,
    "Table 1 Delayed Memory summary row": "\\quad Delayed Memory summary & $-$0.2 (0.8) & 0.3 (0.8) & $-$2.80** & $-$0.57 \\\\" in tex,
    "Table 1 Total Scale row": "\\quad Total Scale (Sum Raw) & 226.3 (18.4) & 238.5 (15.7) & $-$3.55*** & $-$0.70 \\\\" in tex,
    "Abstract Methods dataset sentence": "Revised analyses were performed on an analysis-ready cohort dataset comprising 105 adults (65 IBS, 40 healthy controls) from the Bergen Brain--Gut--Microbiota study." in tex,
    "Abstract Results line updated": f"The pipeline identified clinically significant elevations in sleep ({abstract_summary['sleep_pct']}\\%), fatigue ({abstract_summary['fatigue_pct']}\\%), anxiety ({abstract_summary['anxiety_pct']}\\%), and attention ({abstract_summary['attention_pct']}\\%), with multi-domain flags in {abstract_summary['multi_pct']}\\%." in tex,
    "Sleep count updated": f"{pipeline_counts['sleep_total']} participants (50\\%) triggered a sleep-domain flag" in tex,
    "Attention count updated": f"{pipeline_counts['attention_total']} participants (44\\%) had at least one flagged CPT metric" in tex,
    "Neurocognition count updated": f"{pipeline_counts['cognition_total']} participants (46\\%) met the pipeline's cognition-flag criterion" in tex,
    "Multi-domain count updated": f"The pipeline identified multi-domain elevations (flags in $\\geq 2$ domains) in {pipeline_counts['multi_domain_total']} participants (63\\%), predominantly within the IBS group ($n = {pipeline_counts['multi_domain_ibs']}$)." in tex,
    "Case subj_001 updated": "RBANS Sum Raw = 233 (57th cohort percentile). The raw-subtest-derived profile was broadly mid-range, with no marked cohort-relative cognitive weakness." in tex,
    "Case subj_048 updated": "RBANS (Sum Raw = 213, 15th cohort percentile), with marked relative weakness in Language and additional relative weakness in Immediate Memory, Delayed Memory, and Recognition" in tex,
}

consistency = pd.DataFrame(
    [{"check": name, "passed": passed} for name, passed in checks.items()]
)
display(consistency)

failed = consistency.loc[~consistency["passed"], "check"].tolist()
n_ok = int(consistency["passed"].sum())
print(f"Manuscript consistency: {n_ok}/{len(consistency)} exact-string checks passed "
      f"against {MANUSCRIPT_PATH.name}.")
if failed:
    # These checks are pinned to the verbatim wording/numerics of the original
    # March-16 submission. The manuscript has since been revised (clinical edits
    # in jins_main_rev.tex), so some exact-string matches no longer hold. The
    # underlying pipeline numerics are reproduced in the cells above; this block
    # is therefore reported as a diagnostic rather than a hard failure.
    print("\nChecks no longer matching the current manuscript (expected after "
          "post-submission revision):")
    for name in failed:
        print(f"  - {name}")

## Raw vs cleaned data comparison

This section documents the transition from the raw cohort file `data/BGA_merged_all_20260208.csv` to the technically cleaned file `data/BGA_merged_all_20260208_cleaned.csv`.

The purpose is to make the cleaning process transparent and reproducible while preserving a clear distinction between technical cleaning and later analysis-ready decisions.

Specifically, the comparison examines:

- whether cohort membership and the core variable structure were preserved
- which new columns were added to document data-quality issues
- how explicit missingness changed after blank-string normalization
- which shared variables received non-missing value edits
- how subject-level or domain-level flags were retained in the cleaned file rather than resolved silently

The outputs below combine concise narrative summaries, tables, and visualizations so that the cleaning decisions can be audited and, where useful, reused in project documentation and manuscript text.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

RAW_PATH = ROOT / "data" / "BGA_merged_all_20260208.csv"
CLEANED_PATH = ROOT / "data" / "BGA_merged_all_20260208_cleaned.csv"

raw_text = pd.read_csv(RAW_PATH, sep=";", dtype=str, keep_default_na=False)
cleaned_text = pd.read_csv(CLEANED_PATH, sep=";", dtype=str, keep_default_na=False)
raw_text.columns = raw_text.columns.str.strip()
cleaned_text.columns = cleaned_text.columns.str.strip()

raw_norm = raw_text.replace(r"^\s*$", np.nan, regex=True)
cleaned_norm = cleaned_text.replace(r"^\s*$", np.nan, regex=True)

shared_cols = [c for c in raw_text.columns if c in cleaned_text.columns]
added_cols = [c for c in cleaned_text.columns if c not in raw_text.columns]
removed_cols = [c for c in raw_text.columns if c not in cleaned_text.columns]

blank_counts_raw = raw_text.apply(lambda s: s.astype(str).str.strip().eq("").sum())
blank_counts_cleaned = cleaned_text.apply(lambda s: s.astype(str).str.strip().eq("").sum())
missing_counts_raw = raw_norm.isna().sum()
missing_counts_cleaned = cleaned_norm.isna().sum()

changed_any_counts = {}
changed_nonmissing_counts = {}
blank_to_missing_counts = {}
for col in shared_cols:
    raw_col = raw_norm[col]
    cleaned_col = cleaned_norm[col]
    changed_any_counts[col] = int(raw_col.fillna("<NA>").ne(cleaned_col.fillna("<NA>")).sum())
    changed_nonmissing_counts[col] = int((raw_col.notna() & cleaned_col.notna() & raw_col.ne(cleaned_col)).sum())
    blank_to_missing_counts[col] = int(raw_text[col].astype(str).str.strip().eq("").sum()) - int(cleaned_text[col].astype(str).str.strip().eq("").sum())

structure_df = pd.DataFrame(
    [
        {"file": RAW_PATH.name, "rows": raw_text.shape[0], "columns": raw_text.shape[1]},
        {"file": CLEANED_PATH.name, "rows": cleaned_text.shape[0], "columns": cleaned_text.shape[1]},
    ]
)

column_change_df = pd.DataFrame(
    {
        "shared_columns": [len(shared_cols)],
        "added_in_cleaned": [len(added_cols)],
        "removed_from_cleaned": [len(removed_cols)],
    }
)

missingness_df = pd.DataFrame(
    {
        "column": shared_cols,
        "raw_missing": [int(missing_counts_raw[c]) for c in shared_cols],
        "cleaned_missing": [int(missing_counts_cleaned[c]) for c in shared_cols],
        "missing_delta": [int(missing_counts_cleaned[c] - missing_counts_raw[c]) for c in shared_cols],
        "nonmissing_value_changes": [changed_nonmissing_counts[c] for c in shared_cols],
    }
).sort_values(["nonmissing_value_changes", "missing_delta"], ascending=[False, False])

def parse_flag_series(series: pd.Series) -> pd.Series:
    normalized = series.fillna("").astype(str).str.strip().str.lower()
    return normalized.isin(["1", "true", "yes"])


flag_summary_df = pd.DataFrame()
if added_cols:
    flag_summary_df = pd.DataFrame(
        {
            "flag_column": added_cols,
            "flagged_subjects": [int(parse_flag_series(cleaned_text[c]).sum()) for c in added_cols],
        }
    ).sort_values("flagged_subjects", ascending=False)

summary_md = f"""
### Structural summary

- Raw file shape: `{raw_text.shape[0]}` rows x `{raw_text.shape[1]}` columns
- Cleaned file shape: `{cleaned_text.shape[0]}` rows x `{cleaned_text.shape[1]}` columns
- Shared columns: `{len(shared_cols)}`
- Added columns in cleaned file: `{len(added_cols)}`
- Removed columns: `{len(removed_cols)}`

### Interpretation for documentation and Methods text

- The cleaned dataset preserves cohort membership: the row count is unchanged from the raw source file.
- The core data structure is retained; the main structural change is the addition of cleaning flags that make missing blocks and unresolved score issues explicit.
- Blank-string entries used in the raw CSV are normalized to explicit missing values in the cleaned file, improving reproducibility of downstream summaries and models.
- Non-missing value edits are limited to a small subset of shared variables, which means substantive cleaning decisions can be inspected directly rather than inferred from broad file-level changes.
- In keeping with the cleaning log, the cleaned file documents issues conservatively and does not silently convert it into an analysis-ready dataset.
"""

display(Markdown(summary_md))
display(structure_df)
display(column_change_df)
display(pd.DataFrame({"added_columns": pd.Series(added_cols), "removed_columns": pd.Series(removed_cols)}))
display(missingness_df.head(20))
if not flag_summary_df.empty:
    display(flag_summary_df)

### Variable-level cleaning examples

The next cell highlights concrete variable-level changes that are suitable for reuse in the cleaning log or manuscript Methods.

In particular, it inspects:

- `Education`, where text-like values were standardized to support reproducible numeric summaries
- `HandPref`, where category normalization was applied and documented
- the newly added flag columns, which preserve information about structural missingness or unresolved score discrepancies without dropping participants from the cleaned file

In [ ]:
def unique_nonmissing(series: pd.Series, n: int = 20):
    values = [v for v in series.astype(str).tolist() if str(v).strip() != ""]
    return sorted(set(values))[:n]

handpref_available = "HandPref" in raw_text.columns and "HandPref" in cleaned_text.columns
if handpref_available:
    handpref_examples = pd.DataFrame(
        {
            "raw_unique": pd.Series(unique_nonmissing(raw_text["HandPref"])),
            "cleaned_unique": pd.Series(unique_nonmissing(cleaned_text["HandPref"])),
        }
    )
else:
    handpref_examples = pd.DataFrame(
        {"note": ["`HandPref` is not present in the current cleaned/raw comparison files."]}
    )

education_examples = pd.DataFrame(
    {
        "raw_example_values": pd.Series(unique_nonmissing(raw_text["Education"], n=15)),
        "cleaned_example_values": pd.Series(unique_nonmissing(cleaned_text["Education"], n=15)),
    }
)

column_focus_rows = [
    {
        "column": "Education",
        "raw_blank_strings": int(raw_text["Education"].astype(str).str.strip().eq("").sum()),
        "cleaned_blank_strings": int(cleaned_text["Education"].astype(str).str.strip().eq("").sum()),
        "raw_missing_after_normalization": int(raw_norm["Education"].isna().sum()),
        "cleaned_missing_after_normalization": int(cleaned_norm["Education"].isna().sum()),
        "nonmissing_value_changes": changed_nonmissing_counts.get("Education", 0),
    }
]
if handpref_available:
    column_focus_rows.append(
        {
            "column": "HandPref",
            "raw_blank_strings": int(raw_text["HandPref"].astype(str).str.strip().eq("").sum()),
            "cleaned_blank_strings": int(cleaned_text["HandPref"].astype(str).str.strip().eq("").sum()),
            "raw_missing_after_normalization": int(raw_norm["HandPref"].isna().sum()),
            "cleaned_missing_after_normalization": int(cleaned_norm["HandPref"].isna().sum()),
            "nonmissing_value_changes": changed_nonmissing_counts.get("HandPref", 0),
        }
    )

column_focus_df = pd.DataFrame(column_focus_rows)

flagged_subjects_df = cleaned_text.loc[:, added_cols].copy()
if not flagged_subjects_df.empty:
    flag_matrix = pd.DataFrame({c: parse_flag_series(cleaned_text[c]) for c in added_cols})
    flagged_subjects_df = flag_matrix.copy()
    flagged_subjects_df.insert(0, "Subject", cleaned_text["Subject"])
    flagged_subjects_preview = flagged_subjects_df.loc[
        flag_matrix.sum(axis=1) > 0
    ].head(15)
else:
    flagged_subjects_preview = pd.DataFrame()

focus_md = f"""
### Focused interpretation

- `Education` illustrates technical standardization rather than conceptual recoding: values are prepared for numeric analysis while preserving the recorded years of education.
- `HandPref` is discussed only when present in both files; the current comparison therefore degrades gracefully if that column is absent from the cleaned/raw pair being audited.
- The cleaned file adds `{len(added_cols)}` flag columns to document structural missingness and unresolved data-quality concerns. These flags preserve auditability and support domain-specific inclusion decisions at later analysis stages.
"""

display(Markdown(focus_md))
display(column_focus_df)
display(handpref_examples)
display(education_examples)
if not flagged_subjects_preview.empty:
    display(flagged_subjects_preview)

### Visual comparison

The plots below translate the cleaning log into an immediately inspectable form:

1. the largest changes in explicit missingness after blank-string normalization
2. the shared variables with the largest number of non-missing value edits
3. the prevalence of the added cleaning flags across participants

Together, these visual summaries help distinguish routine standardization from variables that may require closer methodological explanation.

In [ ]:
sns.set_theme(style="whitegrid", palette="colorblind")

plot_missing = missingness_df.loc[missingness_df["missing_delta"] != 0].copy()
plot_missing = plot_missing.sort_values("missing_delta", ascending=False).head(12)

plot_changes = missingness_df.loc[missingness_df["nonmissing_value_changes"] > 0].copy()
plot_changes = plot_changes.sort_values("nonmissing_value_changes", ascending=False).head(12)

fig, axes = plt.subplots(3, 1, figsize=(12, 16), constrained_layout=True)

if not plot_missing.empty:
    sns.barplot(data=plot_missing, x="missing_delta", y="column", ax=axes[0], color="#4C72B0")
    axes[0].set_title("Largest increases in explicit missingness after cleaning")
    axes[0].set_xlabel("cleaned missing - raw missing")
    axes[0].set_ylabel("")
else:
    axes[0].text(0.5, 0.5, "No missingness differences detected", ha="center", va="center")
    axes[0].set_axis_off()

if not plot_changes.empty:
    sns.barplot(data=plot_changes, x="nonmissing_value_changes", y="column", ax=axes[1], color="#DD8452")
    axes[1].set_title("Columns with non-missing value edits")
    axes[1].set_xlabel("Number of edited non-missing entries")
    axes[1].set_ylabel("")
else:
    axes[1].text(0.5, 0.5, "No non-missing value edits detected", ha="center", va="center")
    axes[1].set_axis_off()

if not flag_summary_df.empty:
    sns.barplot(data=flag_summary_df.sort_values("flagged_subjects", ascending=False), x="flagged_subjects", y="flag_column", ax=axes[2], color="#55A868")
    axes[2].set_title("Prevalence of added cleaning flags")
    axes[2].set_xlabel("Flagged participants")
    axes[2].set_ylabel("")
else:
    axes[2].text(0.5, 0.5, "No added flag columns found", ha="center", va="center")
    axes[2].set_axis_off()

for ax in axes:
    ax.tick_params(axis="y", labelsize=9)

fig.suptitle("Comparison of raw and cleaned BGA cohort files", fontsize=14)

from io import BytesIO
from IPython.display import Image

buffer = BytesIO()
fig.savefig(buffer, format="png", dpi=200, bbox_inches="tight")
buffer.seek(0)
display(Image(data=buffer.getvalue()))
plt.close(fig)

## Manuscript-facing comparison: raw versus cleaned data for Table 1 and group differences

This section asks a narrower manuscript question than the earlier file-level audit: do the cohort characteristics and IBS-versus-HC group differences reported in `manuscript/jins_main.tex` change when the analyses are run from the original raw CSV instead of the cleaned CSV?

The comparison below uses the same loading and scoring logic for both files and then re-derives the quantities that feed:

- Table 1 (`Cohort demographics and neuropsychological domain scores by group`)
- the Results subsection `Group differences across domains`

This is useful because the cleaning workflow mainly introduced explicit missingness handling, category normalization, and issue flags. The analyses below test whether those technical changes altered the manuscript-level inferential results or whether the published comparisons are stable across the raw and cleaned versions of the cohort file.

In [ ]:
RAW_DATA_PATH = ROOT / "data" / "BGA_merged_all_20260208.csv"
CLEANED_DATA_PATH = ROOT / "data" / "BGA_merged_all_20260208_cleaned.csv"
ANALYSIS_READY_PATH = Path(pipeline.DATA_PATH)

comparison_md = ""
try:
    _ = pipeline.load_cohort(str(RAW_DATA_PATH))
    _ = pipeline.load_cohort(str(CLEANED_DATA_PATH))
except Exception as exc:
    comparison_md = f"""
### Summary of manuscript-facing stability

The active March 16 standalone pipeline expects the analysis-ready schema in `{ANALYSIS_READY_PATH.name}` and therefore cannot be applied directly to the historical raw/cleaned files used in this older comparison block.

- Raw file: `{RAW_DATA_PATH.name}`
- Technically cleaned file: `{CLEANED_DATA_PATH.name}`
- Active pipeline reference file: `{ANALYSIS_READY_PATH.name}`
- Reason direct raw-vs-cleaned Table 1 recomputation is skipped here: `{exc}`

### Interpretation

This notebook now treats the raw-vs-cleaned section as historical context only. The manuscript-facing statistics are reproduced from the active analysis-ready file and validated against `manuscript/jins_main.tex` in the notebook cells above.
"""
    display(Markdown(comparison_md))
else:
    comparison_md = """
### Summary of manuscript-facing stability

The raw and technically cleaned files unexpectedly matched the active analysis-ready schema. If this occurs in a future revision, a direct raw-vs-cleaned recomputation block can be reinstated here.
"""
    display(Markdown(comparison_md))

### Visual comparison of manuscript-level stability

The figures below compare the raw-file and cleaned-file versions of the manuscript statistics directly.

They are designed to answer two practical questions:

1. Do the Table 1 effect sizes and p-values move after cleaning?
2. Do the domain-level conclusions reported in the Results section change?

If the points fall on the identity line and the overlays coincide, then the manuscript-level group comparisons are empirically stable across the two file versions even though the cleaned file improves documentation and auditability.

In [ ]:
sns.set_theme(style="whitegrid", palette="colorblind")

if "raw_vs_clean_table1" not in globals():
    display(Markdown(
        "Raw-versus-cleaned comparison plots are skipped because the current March 16 pipeline only supports the analysis-ready schema, not the historical raw/cleaned file pair."
    ))
else:
    plot_df = raw_vs_clean_table1.copy()
    plot_df["neglog10_p_raw"] = -np.log10(plot_df["p_raw"].clip(lower=1e-12))
    plot_df["neglog10_p_clean"] = -np.log10(plot_df["p_clean"].clip(lower=1e-12))
    plot_df["abs_d_clean"] = plot_df["d_clean"].abs()
    plot_df = plot_df.sort_values(["Domain", "abs_d_clean"], ascending=[True, False])

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)

    max_d = float(np.ceil(max(plot_df["d_raw"].abs().max(), plot_df["d_clean"].abs().max()) * 10) / 10)
    axes[0].scatter(plot_df["d_raw"], plot_df["d_clean"], s=70, color="#4C72B0")
    axes[0].plot([-max_d, max_d], [-max_d, max_d], linestyle="--", color="black", linewidth=1)
    for _, row in plot_df.iterrows():
        axes[0].text(row["d_raw"], row["d_clean"], row["Measure"], fontsize=8, ha="left", va="bottom")
    axes[0].set_title("Table 1 effect sizes: raw versus cleaned")
    axes[0].set_xlabel("Cohen's d from raw file")
    axes[0].set_ylabel("Cohen's d from cleaned file")
    axes[0].set_xlim(-max_d - 0.1, max_d + 0.1)
    axes[0].set_ylim(-max_d - 0.1, max_d + 0.1)

    max_p = float(np.ceil(max(plot_df["neglog10_p_raw"].max(), plot_df["neglog10_p_clean"].max()) * 10) / 10)
    axes[1].scatter(plot_df["neglog10_p_raw"], plot_df["neglog10_p_clean"], s=70, color="#55A868")
    axes[1].plot([0, max_p], [0, max_p], linestyle="--", color="black", linewidth=1)
    for _, row in plot_df.iterrows():
        axes[1].text(row["neglog10_p_raw"], row["neglog10_p_clean"], row["Measure"], fontsize=8, ha="left", va="bottom")
    axes[1].axvline(-np.log10(0.05), linestyle=":", color="#C44E52", linewidth=1)
    axes[1].axhline(-np.log10(0.05), linestyle=":", color="#C44E52", linewidth=1)
    axes[1].set_title("Welch test p-values: raw versus cleaned")
    axes[1].set_xlabel("-log10(p) from raw file")
    axes[1].set_ylabel("-log10(p) from cleaned file")
    axes[1].set_xlim(-0.1, max_p + 0.2)
    axes[1].set_ylim(-0.1, max_p + 0.2)

    fig.suptitle("Stability of manuscript group comparisons across raw and cleaned BGA files", fontsize=14)

    from io import BytesIO
    from IPython.display import Image

    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=200, bbox_inches="tight")
    buffer.seek(0)
    display(Image(data=buffer.getvalue()))
    plt.close(fig)

    display(Markdown(
        "**Take-home message.** In both scatterplots, every point lies on the identity line, showing that the raw and cleaned files yield the same effect sizes and the same p-values for all Table 1 measures. This means the manuscript's IBS-versus-HC group comparisons are empirically unchanged by the cleaning step."
    ))

    fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
    measure_order = plot_df["Measure"].tolist()
    y_pos = np.arange(len(measure_order))
    ax.hlines(y=y_pos, xmin=plot_df["d_raw"], xmax=plot_df["d_clean"], color="#BBBBBB", linewidth=2)
    ax.scatter(plot_df["d_raw"], y_pos, color="#DD8452", label="Raw", s=55)
    ax.scatter(plot_df["d_clean"], y_pos, color="#4C72B0", marker="s", label="Cleaned", s=45)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(measure_order)
    ax.set_xlabel("Cohen's d (IBS - HC)")
    ax.set_title("Domain-wise effect sizes reported in Table 1")
    ax.legend(frameon=True)
    ax.invert_yaxis()

    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=200, bbox_inches="tight")
    buffer.seek(0)
    display(Image(data=buffer.getvalue()))
    plt.close(fig)

### Interpretation of the effect-size figure

This figure shows that the raw-file and cleaned-file effect sizes are visually superimposed for every Table 1 measure. In practical terms, the cleaning steps did not change the magnitude or direction of the IBS-versus-HC differences reported in the manuscript.

Several substantive patterns remain clear:

- The largest positive IBS-minus-HC effects are still seen for fatigue (`Chalder total`), anxiety, insomnia (`BIS total`), and depression.
- The largest negative IBS-minus-HC effects are still seen for `Total Scale`, `Immediate Memory`, `Delayed Memory`, and `Attention`, indicating lower RBANS performance in the IBS group on these measures.
- Measures close to zero, such as `HRT`, `Visuospatial`, and `Language`, remain close to zero in both versions of the dataset, supporting the conclusion that these group differences are minimal or absent.

Because the raw and cleaned estimates coincide, this figure supports a strong reproducibility conclusion: for the outcomes currently summarized in Table 1 and discussed in `Group differences across domains`, the cleaned dataset improves transparency and auditability without materially altering the manuscript's inferential results.

## Technical handling of the eight `flag_` columns

The cleaned cohort file should remain a **technically cleaned** dataset rather than a final analysis-ready dataset. In this workflow, the eight trailing `flag_` columns are treated as analysis metadata: they document structural missingness, interpretation constraints, or unresolved score discrepancies, but they should not trigger wholesale row deletion.

The practical implication is that the flags should be used in a **domain-specific** way:

- `flag_missing_bis_block`, `flag_missing_fatigue_block`, `flag_missing_hads_block`, `flag_missing_cpt_block`, and `flag_missing_rbans_block` define domain-specific inclusion masks rather than participant-level exclusion.
- `flag_hc_with_ibs_sss` is an interpretation flag: HC values can remain recorded in the cleaned file, but they should not be used as formal IBS-severity outcomes in an analysis-ready layer.
- `flag_rbans_sum_mismatch` and `flag_tfs_vs_13_item_sum_mismatch` identify variables that should be preserved in the cleaned file but either excluded from affected analyses or routed to separate analysis variables in the analysis-ready file.

This means the recommended next step is not to alter `BGA_merged_all_20260208_cleaned.csv` directly, but to build `BGA_merged_all_20260208_analysis_ready.csv` from it using explicit `use_*` masks and analysis-specific columns.

For the current cleaned file, the flag counts are:

- `flag_missing_bis_block`: 8
- `flag_missing_fatigue_block`: 21
- `flag_missing_hads_block`: 12
- `flag_missing_cpt_block`: 3
- `flag_missing_rbans_block`: 3
- `flag_hc_with_ibs_sss`: 34
- `flag_rbans_sum_mismatch`: 2
- `flag_tfs_vs_13_item_sum_mismatch`: 50

The construction cell below implements this conservative policy by keeping all original cleaned variables, normalizing the flag columns to booleans, adding analysis masks, creating analysis-specific columns for variables that require stricter use rules, and exporting a separate analysis-ready CSV.

In [ ]:
ANALYSIS_READY_PATH = Path(pipeline.DATA_PATH)
LEGACY_ANALYSIS_READY_PATH = ROOT / "data" / "BGA_merged_all_20260208_analysis_ready.csv"

analysis_ready_md = f"""
### Analysis-ready construction summary

The current March 16 workflow no longer constructs a provisional `analysis_ready.csv` inside this notebook. Instead, the standalone pipeline reads the finalized analysis-ready file directly:

- Active pipeline input: `{ANALYSIS_READY_PATH.name}`
- Historical notebook export target: `{LEGACY_ANALYSIS_READY_PATH.name}`
- Technical cleaned source file referenced in older notebook sections: `{CLEANED_DATA_PATH.name}`

This section is therefore retained as documentation only. The active manuscript-facing computations above already use `{ANALYSIS_READY_PATH.name}` through `scripts/neuropsych_pipeline.py`.
"""

display(Markdown(analysis_ready_md))

## Implications for future manuscript revision

The current notebook and data workflow support a conservative interpretation of the eight `flag_` columns: they are best treated as **analysis-routing metadata** rather than as automatic participant-exclusion triggers. Under that approach, the primary effect on the manuscript is expected to be methodological clarification rather than wholesale numerical revision.

More specifically:

- If the flags continue to be used **locally** through domain-specific or variable-specific masks, then future updates to `manuscript/jins_main.tex` will mainly involve clearer Methods wording about how flagged values were retained, masked, or interpreted conservatively.
- Under that same local-masking approach, Table 1 and the subsection `Group differences across domains` are currently stable: the notebook's raw-versus-cleaned comparison showed that the reported IBS-versus-HC effects do not change for the variables currently summarized in the manuscript.
- In contrast, if the flags are later used **aggressively** to redefine the analytic cohort or to exclude broader sets of observations, then the manuscript would need full numerical revision. In that scenario, Table 1, the group-difference paragraph, pipeline summary counts, and possibly abstract percentages would all need to be recalculated and rewritten.

For that reason, the immediate recommendation is to keep the current manuscript untouched until the flag policy is finalized. Once those rules are fixed, the manuscript can be updated in a single, internally consistent pass using the corresponding analysis-ready outputs.